# Day 34 · The bug when a value is used twice

**First-principles question:** What happens when one number feeds two places?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

Code in **`lab/micrograd.py`**: fix every `_backward` to *add* (`+=`), then the step 5 TODOs
(`__pow__`, `exp`, `__truediv__`).

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import math
import lab.micrograd as micrograd
importlib.reload(micrograd)          # picks up your latest edits to lab/micrograd.py
from lab.micrograd import Value, draw

## 1 · The bug
One value used twice. What should `a.grad` be?

In [ ]:
a = Value(3.0, label="a")
b = a + a; b.label = "b"
b.backward()
print("a.grad =", a.grad, "(it should be 2: a pushes into b twice)")

## 2 · New operations

In [ ]:
x = Value(2.0)
print((x ** 3).data, (x.exp()).data, (Value(6.0) / Value(3.0)).data, (x - 5).data, (1 - x).data)

## 3 · tanh built from exp
tanh(n) = (e^{2n} − 1) / (e^{2n} + 1). Same neuron as day 4, different operations: are the gradients the same?

In [ ]:
def neuron(use_exp):
    x1 = Value(2.0); x2 = Value(0.0); w1 = Value(-3.0); w2 = Value(1.0); bias = Value(6.8813735870195432)
    n = x1 * w1 + x2 * w2 + bias
    if use_exp:
        e2 = (2 * n).exp()
        o = (e2 - 1) / (e2 + 1)
    else:
        o = n.tanh()
    o.backward()
    return o.data, [v.grad for v in (x1, x2, w1, w2)]

print("tanh():  ", neuron(False))
print("from exp:", neuron(True))

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert a.grad == 2.0, "Use += in every _backward (step 1)"
assert (Value(2.0) ** 3).data == 8.0 and abs(Value(1.0).exp().data - math.e) < 1e-12
assert (Value(6.0) / Value(3.0)).data == 2.0
p = Value(3.0); q = p ** 2; q.backward(); assert p.grad == 6.0, "d(p²)/dp = 2p"
r = Value(0.5); s = r.exp(); s.backward(); assert abs(r.grad - math.exp(0.5)) < 1e-12, "d(e^r)/dr = e^r"
o1, g1 = neuron(False); o2, g2 = neuron(True)
assert abs(o1 - o2) < 1e-9 and all(abs(u - v) < 1e-9 for u, v in zip(g1, g2)), "tanh from exp should match tanh()"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")